In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# Localiza a raiz do projeto e o diretório da camada bronze
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
pasta_dolar = BASE_DIR / "data" / "bronze" / "dolar"

arquivos = list(pasta_dolar.glob("*.json"))
assert len(arquivos) > 0, "Nenhum arquivo JSON encontrado em data/bronze/dolar/"

arquivo_alvo = arquivos[0]
df_raw = pd.read_json(arquivo_alvo)

print(f"Arquivo carregado: {arquivo_alvo.name}")
print(f"Dimensões: {df_raw.shape[0]} registros e {df_raw.shape[1]} colunas")
df_raw.head()

Arquivo carregado: dolar_2026-09-06.json
Dimensões: 251 registros e 2 colunas


,data,valor
0,08/09/2025,5.4272
1,09/09/2025,5.4272
2,10/09/2025,5.4117
3,11/09/2025,5.3852
4,12/09/2025,5.3671


In [3]:
df = df_raw.copy()

df = df.rename(columns={"valor": "cotacao_dolar"})

# Parsing de data e conversão numérica
df["data"] = pd.to_datetime(df["data"], format="%d/%m/%Y")
df["cotacao_dolar"] = pd.to_numeric(df["cotacao_dolar"], errors="coerce")

# Ordenação cronológica obrigatória para séries temporais
df = df.sort_values("data").reset_index(drop=True)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 251 entries, 0 to 250
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   data           251 non-null    datetime64[us]
 1   cotacao_dolar  251 non-null    float64       
dtypes: datetime64[us](1), float64(1)
memory usage: 4.1 KB


In [4]:
# 1. Unicidade de Data (Primary Key)
assert not df["data"].duplicated().any(), "Erro crítico: Datas duplicadas detectadas!"

# 2. Ausência de Nulos
assert df["data"].isnull().sum() == 0, "Erro crítico: Nulos encontrados na coluna 'data'!"
assert df["cotacao_dolar"].isnull().sum() == 0, "Erro crítico: Nulos encontrados na coluna 'cotacao_dolar'!"

# 3. Consistência Numérica
assert (df["cotacao_dolar"] > 0).all(), "Erro crítico: Cotação do dólar menor ou igual a zero!"

print("Auditoria de integridade aprovada com sucesso.")
display(df["cotacao_dolar"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).to_frame().T)

Auditoria de integridade aprovada com sucesso.


,count,mean,std,min,1%,5%,50%,95%,99%,max
cotacao_dolar,251.0,5.223886,0.15224,4.8967,4.90525,4.98015,5.2092,5.45935,5.5395,5.5733


In [5]:
df["gap_dias"] = df["data"].diff().dt.days
maiores_gaps = df[df["gap_dias"] > 3][["data", "cotacao_dolar", "gap_dias"]]

print(f"Total de períodos com mais de 3 dias sem cotação: {len(maiores_gaps)}")
if not maiores_gaps.empty:
    display(maiores_gaps.sort_values("gap_dias", ascending=False).head(5))

df.drop(columns=["gap_dias"], inplace=True)

Total de períodos com mais de 3 dias sem cotação: 3


,data,cotacao_dolar,gap_dias
112,2026-02-18,5.2343,5.0
144,2026-04-06,5.1526,4.0
162,2026-05-04,4.9581,4.0


In [6]:
# Médias móveis
df["dolar_mm7d"] = df["cotacao_dolar"].rolling(window=7).mean().round(4)
df["dolar_mm30d"] = df["cotacao_dolar"].rolling(window=30).mean().round(4)

# Amostra da consolidação mensal
df_mensal = df.groupby(df["data"].dt.to_period("M")).agg(
    dolar_medio=("cotacao_dolar", "mean"),
    dias_negociados=("cotacao_dolar", "count")
).reset_index()

df_mensal["data_referencia"] = df_mensal["data"].dt.to_timestamp()
df_mensal = df_mensal[["data_referencia", "dolar_medio", "dias_negociados"]]

display(df_mensal.tail(6))

,data_referencia,dolar_medio,dias_negociados
7,2026-04-01,5.032475,20
8,2026-05-01,4.983100,20
9,2026-06-01,5.126971,21
10,2026-07-01,5.113343,23
11,2026-08-01,5.152562,21
12,2026-09-01,5.125850,4


In [7]:
# Define e cria o diretório data/silver
pasta_silver = BASE_DIR / "data" / "silver"
pasta_silver.mkdir(parents=True, exist_ok=True)

# Mantém apenas as colunas estruturais do staging para o arquivo Silver
df_silver = df[["data", "cotacao_dolar"]].copy()

# Persistência em Parquet
caminho_teste_parquet = pasta_silver / "dolar_silver_test.parquet"
df_silver.to_parquet(caminho_teste_parquet, index=False)

print("Arquivo Parquet da camada Silver gerado com sucesso!")
print(f"Caminho: {caminho_teste_parquet.relative_to(BASE_DIR)}")

Arquivo Parquet da camada Silver gerado com sucesso!
Caminho: data/silver/dolar_silver_test.parquet
